In [ ]:
import os
import math
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
DATA_ROOT = r"data\CelebFaces"
IMAGE_SIZE = 64
BATCH_SIZE = 64
LATENT_DIM = 128
EPOCHS = 20
LR = 2e-4
BETA = 1.0  # коэффициент KL в beta-VAE
NUM_WORKERS = 0
SEED = 42
MAX_IMAGES = 2000  

CHECKPOINT_DIR = "checkpoints_vae"
SAMPLES_DIR = "samples_vae"

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
class CelebFacesDataset(Dataset):
    def __init__(self, root_dir: str, transform=None, max_images=None, seed=42):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

        if not self.root_dir.exists():
            raise FileNotFoundError(f"Dataset path not found: {self.root_dir}")

        self.image_paths = [
            p for p in self.root_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in self.extensions
        ]


        if max_images is not None:
            max_images = int(max_images)
            if max_images <= 0:
                raise ValueError("max_images must be > 0 or None")
            if max_images < len(self.image_paths):
                rng = random.Random(seed)
                self.image_paths = rng.sample(self.image_paths, k=max_images)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

dataset = CelebFacesDataset(
    DATA_ROOT,
    transform=transform,
    max_images=MAX_IMAGES,
    seed=SEED,
)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Dataset size: {len(dataset)} images")

batch = next(iter(dataloader))
grid = make_grid(batch[:16], nrow=4)
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("Sample images")
plt.show()

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder: 3x64x64 -> 256x4x4
        self.encoder = nn.Sequential(
            # Напишите свой энкодер исходя из условий
        )

        self.flatten_dim = 256 * 4 * 4
        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_dim, latent_dim)

        self.fc_decode = nn.Linear(latent_dim, self.flatten_dim)

        # Decoder: 256x4x4 -> 3x64x64
        self.decoder = nn.Sequential(
            # Напишите свой декодер исходя из условий
        )

    def encode(self, x):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(-1, 256, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

In [ ]:
def vae_loss(x_recon, x, mu, logvar, beta=1.0):
    # напишите свою функцию потерь
    pass


@torch.no_grad()
def plot_reconstructions(model, batch, epoch, out_dir):
    model.eval()
    batch = batch.to(device)
    recon, _, _ = model(batch)

    n = min(8, batch.size(0))
    comparison = torch.cat([batch[:n], recon[:n]], dim=0)
    grid = make_grid(comparison, nrow=n)

    plt.figure(figsize=(2 * n, 4))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
    plt.axis("off")
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def plot_samples(model, epoch, out_dir, latent_dim=128, n=16):
    model.eval()
    z = torch.randn(n, latent_dim, device=device)
    samples = model.decode(z)

    grid = make_grid(samples, nrow=int(math.sqrt(n)))
    plt.figure(figsize=(6, 6))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
model = ConvVAE(latent_dim=LATENT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(model)

In [ ]:
history = {
    "loss": [],
    "recon": [],
    "kl": [],
}

if "dataloader" not in globals():
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
    ])
    dataset = CelebFacesDataset(
        DATA_ROOT,
        transform=transform,
        max_images=MAX_IMAGES,
        seed=SEED,
    )
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

fixed_batch = next(iter(dataloader))[:16]

for epoch in range(1, EPOCHS + 1):
    model.train()

    epoch_loss = 0.0
    epoch_recon = 0.0
    epoch_kl = 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}")
    for x in pbar:
        x = x.to(device)

        optimizer.zero_grad()
        x_recon, mu, logvar = model(x)
        loss, recon_loss, kl_loss = vae_loss(x_recon, x, mu, logvar, beta=BETA)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_recon += recon_loss.item()
        epoch_kl += kl_loss.item()

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            recon=f"{recon_loss.item():.4f}",
            kl=f"{kl_loss.item():.4f}",
        )

    num_batches = len(dataloader)
    avg_loss = epoch_loss / num_batches
    avg_recon = epoch_recon / num_batches
    avg_kl = epoch_kl / num_batches

    history["loss"].append(avg_loss)
    history["recon"].append(avg_recon)
    history["kl"].append(avg_kl)

    print(
        f"Epoch {epoch}/{EPOCHS}"
        f"loss: {avg_loss:.4f}  recon: {avg_recon:.4f}  kl: {avg_kl:.4f}"
    )

    ckpt_path = f"vae_epoch_{epoch:03d}.pt"
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "config": {
                "image_size": IMAGE_SIZE,
                "batch_size": BATCH_SIZE,
                "latent_dim": LATENT_DIM,
                "lr": LR,
                "beta": BETA,
            },
        },
        ckpt_path,
    )

    plot_reconstructions(model, fixed_batch, epoch, SAMPLES_DIR)
    plot_samples(model, epoch, SAMPLES_DIR, latent_dim=LATENT_DIM, n=16)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history["loss"], label="total")
plt.plot(history["recon"], label="recon")
plt.plot(history["kl"], label="kl")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training curves")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:


transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])
dataset = CelebFacesDataset(
    DATA_ROOT,
    transform=transform,
    max_images=MAX_IMAGES,  
    seed=SEED,
)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model.eval()

test_batch = next(iter(dataloader)).to(device)

with torch.no_grad():
    test_recon, test_mu, test_logvar = model(test_batch)
    test_loss, test_recon_loss, test_kl_loss = vae_loss(
        test_recon, test_batch, test_mu, test_logvar, beta=BETA
    )

n = min(8, test_batch.size(0))
comparison = torch.cat([test_batch[:n], test_recon[:n]], dim=0)
grid = make_grid(comparison, nrow=n)

plt.figure(figsize=(2 * n, 4))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.tight_layout()
plt.show()